In [2]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [3]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [4]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [5]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [7]:
rec = answers[0]

In [8]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [ ]:
# calling the llm judge
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation, #calling the class
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, but certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

In [10]:
calc_price(usage)

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.0002295,
 'total_cost': 0.00045225}

In [11]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [ ]:
# inputting the question and answer we want to evaluate
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key meaning of the ground truth: late joining is allowed, but certificate eligibility depends on submitting the project while submissions are still open. It is semantically equivalent.', score='good')

In [ ]:
# run the eval on all answers
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [14]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/395 [00:00<?, ?it/s]

In [15]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [16]:
df_eval = pd.DataFrame(evaluations)

In [17]:
calc_total_price(usages)

0.25354950000000004

In [18]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 378/395 = 95.70%


In [19]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
15,How do the free GPU hours work on these cloud ...,c6c2888275,bad,The AI answer fails to answer the question and...
29,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer contradicts the ground truth. Th...
38,How will I know when a module is actually read...,96286b4be4,bad,The AI answer does not convey the ground-truth...
106,Do I need an OpenAI API key just to check how ...,fe8fed31e6,bad,The AI answer does not answer the question and...
124,What model or provider should I switch to if I...,6e1d8a7b29,bad,The AI answer does not answer the question and...


In [21]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)